# Lecture 7 — Tutorial 3.45: Applied Data Analysis, Visualisation and Interpretation Problem Solving

This notebook contains model solutions for the ten applied Lecture 7 problems. It reuses the curated versions of the Lecture 6 datasets so that the workflow remains continuous from data quality to analysis.

Try each website problem before reading the model solution. The model is one defensible route, not the only possible analytical route.

**Numbering rule:** each problem keeps the same identifier used on the tutorial website. This notebook does not use a separate local problem numbering scheme.

## Problem 3.45.1 — Service demand and reported experience

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_cleaned.csv"
df = pd.read_csv(url)
df["satisfaction_score_clean"] = pd.to_numeric(df["satisfaction_score_clean"], errors="coerce")

# Convert the cleaned Yes/No value into a numeric indicator for a rate.
df["repeat_contact_indicator"] = df["repeat_contact_clean"].map({"Yes":1, "No":0})

summary = (
    df.groupby("service_type", as_index=False)
      .agg(
          request_count=("request_id","count"),
          valid_satisfaction_n=("satisfaction_score_clean","count"),
          mean_satisfaction=("satisfaction_score_clean","mean"),
          median_satisfaction=("satisfaction_score_clean","median"),
          repeat_contact_n=("repeat_contact_indicator","count"),
          repeat_contact_rate=("repeat_contact_indicator","mean"),
      )
)
summary["request_share_pct"] = summary["request_count"] / summary["request_count"].sum() * 100
summary["repeat_contact_rate_pct"] = summary["repeat_contact_rate"] * 100
print(summary.sort_values("request_count", ascending=False).to_string(index=False))

plt.figure(figsize=(8,5))
plot1 = summary.sort_values("request_count", ascending=True)
plt.barh(plot1["service_type"], plot1["request_count"])
plt.xlabel("Number of requests")
plt.ylabel("Service type")
plt.title("Municipal service request volume by service type")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plot2 = summary.sort_values("mean_satisfaction", ascending=True)
plt.barh(plot2["service_type"], plot2["mean_satisfaction"])
plt.xlabel("Mean satisfaction score (valid 1–5 ratings only)")
plt.ylabel("Service type")
plt.title("Reported satisfaction by service type")
plt.xlim(1,5)
plt.tight_layout()
plt.show()

## Problem 3.45.2 — Channel, resolution time and satisfaction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_cleaned.csv"
df = pd.read_csv(url)
df["resolution_hours_clean"] = pd.to_numeric(df["resolution_hours_clean"], errors="coerce")
df["satisfaction_score_clean"] = pd.to_numeric(df["satisfaction_score_clean"], errors="coerce")

# Filter only invalid resolution records for the time comparison.
time_df = df[(df["flag_invalid_resolution"] == "No") & df["resolution_hours_clean"].notna()].copy()
time_summary = time_df.groupby("channel", as_index=False).agg(
    n=("resolution_hours_clean","count"),
    mean_hours=("resolution_hours_clean","mean"),
    median_hours=("resolution_hours_clean","median"),
)
print("OVERALL CHANNEL TIME SUMMARY")
print(time_summary.to_string(index=False))

sat_df = df[(df["flag_invalid_satisfaction"] == "No") & df["satisfaction_score_clean"].notna()].copy()
sat_summary = sat_df.groupby("channel", as_index=False).agg(
    n=("satisfaction_score_clean","count"),
    mean_satisfaction=("satisfaction_score_clean","mean"),
)
print("\nOVERALL CHANNEL SATISFACTION")
print(sat_summary.to_string(index=False))

# Repeat the time comparison inside one service type to reduce service-mix differences.
focus_service = "Waste Collection"
within = time_df[time_df["service_type"] == focus_service]
within_summary = within.groupby("channel", as_index=False).agg(
    n=("resolution_hours_clean","count"), median_hours=("resolution_hours_clean","median")
)
print(f"\nWITHIN {focus_service}")
print(within_summary.to_string(index=False))

plt.figure(figsize=(7,4))
plt.bar(time_summary["channel"], time_summary["median_hours"])
plt.ylabel("Median resolution hours")
plt.xlabel("Channel")
plt.title("Median resolution time by service channel")
plt.tight_layout(); plt.show()

plt.figure(figsize=(7,4))
plt.bar(sat_summary["channel"], sat_summary["mean_satisfaction"])
plt.ylabel("Mean satisfaction (1–5)")
plt.xlabel("Channel")
plt.title("Reported satisfaction by service channel")
plt.ylim(1,5)
plt.tight_layout(); plt.show()

## Problem 3.45.3 — Themes in service feedback

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/service_requests_cleaned.csv"
df = pd.read_csv(url)
df["feedback_clean"] = df["feedback_clean"].fillna("").astype(str)

# Comment availability by service type.
availability = df.assign(has_feedback=df["feedback_clean"].str.strip().ne("")).groupby("service_type").agg(
    requests=("request_id","count"), usable_feedback=("has_feedback","sum")
)
availability["feedback_availability_pct"] = availability["usable_feedback"] / availability["requests"] * 100
print(availability)

# Overall frequency count.
all_tokens = " ".join(df.loc[df["feedback_clean"].str.strip().ne(""), "feedback_clean"]).split()
overall = Counter(all_tokens)
top = pd.DataFrame(overall.most_common(15), columns=["word","count"])
print("\nTOP WORDS")
print(top)

plt.figure(figsize=(8,5))
plot_top = top.sort_values("count")
plt.barh(plot_top["word"], plot_top["count"])
plt.xlabel("Word count")
plt.ylabel("Prepared word")
plt.title("Most frequent prepared words in service feedback")
plt.tight_layout(); plt.show()

# Normalised comparison for two services.
services = ["Waste Collection", "Citizen ID"]
for service in services:
    text = " ".join(df.loc[(df["service_type"] == service) & df["feedback_clean"].str.strip().ne(""), "feedback_clean"])
    tokens = text.split()
    counts = Counter(tokens)
    total = len(tokens)
    table = pd.DataFrame(counts.most_common(10), columns=["word","count"])
    table["per_1000_tokens"] = table["count"] / total * 1000 if total else 0
    print(f"\n{service} — top terms per 1,000 tokens")
    print(table)

# Return from word counts to original records.
selected_terms = ["wait", "clear", "not"]
pattern = "|".join(selected_terms)
examples = df[df["feedback_raw"].fillna("").str.lower().str.contains(pattern, regex=True)]
print("\nORIGINAL FEEDBACK FOR CONTEXT")
print(examples[["service_type","feedback_raw"]].head(20).to_string(index=False))

## Problem 3.45.4 — Waiting time and satisfaction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_cleaned.csv"
df = pd.read_csv(url)
df["wait_minutes_clean"] = pd.to_numeric(df["wait_minutes_clean"], errors="coerce")
df["satisfaction_score_clean"] = pd.to_numeric(df["satisfaction_score_clean"], errors="coerce")

valid = df[(df["flag_invalid_wait"] == "No") & (df["flag_invalid_satisfaction"] == "No")].dropna(subset=["wait_minutes_clean","satisfaction_score_clean"]).copy()
print(valid[["wait_minutes_clean","satisfaction_score_clean"]].describe())

plt.figure(figsize=(7,5))
plt.scatter(valid["wait_minutes_clean"], valid["satisfaction_score_clean"], alpha=0.45)
plt.xlabel("Waiting time (minutes)")
plt.ylabel("Satisfaction score (1–5)")
plt.title("Waiting time and reported satisfaction")
plt.tight_layout(); plt.show()

corr = valid[["wait_minutes_clean","satisfaction_score_clean"]].corr().iloc[0,1]
print("Pearson correlation:", round(corr,3), "Valid N:", len(valid))

# Create transparent wait-time bands.
bins=[-0.001,5,15,30,float("inf")]
labels=["0–5","6–15","16–30","31+"]
valid["wait_band"] = pd.cut(valid["wait_minutes_clean"], bins=bins, labels=labels)
band_summary = valid.groupby("wait_band", observed=False).agg(
    n=("satisfaction_score_clean","count"),
    mean_satisfaction=("satisfaction_score_clean","mean"),
    median_satisfaction=("satisfaction_score_clean","median"),
).reset_index()
print("\nWAIT BAND SUMMARY")
print(band_summary)

plt.figure(figsize=(7,4))
plt.bar(band_summary["wait_band"].astype(str), band_summary["mean_satisfaction"])
plt.xlabel("Waiting-time band (minutes)")
plt.ylabel("Mean satisfaction (1–5)")
plt.title("Satisfaction by waiting-time band")
plt.ylim(1,5)
plt.tight_layout(); plt.show()

## Problem 3.45.5 — Resolution and follow-up by digital-support topic

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_cleaned.csv"
df = pd.read_csv(url)

df["resolved_indicator"] = df["resolved_clean"].map({"Yes":1,"No":0})
df["followup_indicator"] = df["follow_up_needed_clean"].map({"Yes":1,"No":0})

summary = df.groupby("support_topic", as_index=False).agg(
    sessions=("session_id","count"),
    valid_resolution_n=("resolved_indicator","count"),
    resolved_count=("resolved_indicator","sum"),
    resolution_rate=("resolved_indicator","mean"),
    valid_followup_n=("followup_indicator","count"),
    followup_count=("followup_indicator","sum"),
    followup_rate=("followup_indicator","mean"),
)
summary["resolution_rate_pct"] = summary["resolution_rate"]*100
summary["followup_rate_pct"] = summary["followup_rate"]*100
print(summary.to_string(index=False))

plt.figure(figsize=(8,5))
plot = summary.sort_values("resolution_rate_pct")
plt.barh(plot["support_topic"], plot["resolution_rate_pct"])
plt.xlabel("Resolution rate (%)")
plt.ylabel("Support topic")
plt.title("Resolution rate by digital-support topic")
plt.xlim(0,100)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,5))
plot = summary.sort_values("sessions")
plt.barh(plot["support_topic"], plot["sessions"])
plt.xlabel("Number of sessions")
plt.ylabel("Support topic")
plt.title("Digital-support session volume by topic")
plt.tight_layout(); plt.show()

# Within one high-volume topic, compare access modes.
focus = summary.sort_values("sessions", ascending=False).iloc[0]["support_topic"]
within = df[df["support_topic"] == focus].groupby("access_mode").agg(
    n=("resolved_indicator","count"), resolution_rate=("resolved_indicator","mean")
)
print(f"\n{focus} by access mode")
print(within)

## Problem 3.45.6 — Change over reporting months

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/digital_support_sessions_cleaned.csv"
df = pd.read_csv(url)
df["wait_minutes_clean"] = pd.to_numeric(df["wait_minutes_clean"], errors="coerce")

# Keep rows with a usable month for time summaries.
time_df = df[df["report_month"].notna() & (df["report_month"].astype(str).str.len() == 7)].copy()
monthly = time_df.groupby("report_month", as_index=False).agg(
    session_count=("session_id","count"),
    valid_wait_n=("wait_minutes_clean","count"),
    median_wait=("wait_minutes_clean","median"),
)
monthly = monthly.sort_values("report_month")
print(monthly)

plt.figure(figsize=(8,4))
plt.plot(monthly["report_month"], monthly["session_count"], marker="o")
plt.xlabel("Reporting month")
plt.ylabel("Number of sessions")
plt.title("Digital-support session volume over time")
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(monthly["report_month"], monthly["median_wait"], marker="o")
plt.xlabel("Reporting month")
plt.ylabel("Median wait (minutes)")
plt.title("Median digital-support waiting time over time")
plt.xticks(rotation=45)
plt.tight_layout(); plt.show()

# Compare two cities so the overall trend is not treated as universal.
city_month = time_df.groupby(["city","report_month"], as_index=False).agg(
    n=("session_id","count"), median_wait=("wait_minutes_clean","median")
)
for city in ["Copenhagen","Aalborg"]:
    sub=city_month[city_month["city"]==city]
    print(f"\n{city}")
    print(sub.to_string(index=False))

## Problem 3.45.7 — Manual, automated and AI-assisted task time

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/automation_pilot_cleaned.csv"
df = pd.read_csv(url)
df["minutes_spent_clean"] = pd.to_numeric(df["minutes_spent_clean"], errors="coerce")
valid = df[(df["flag_invalid_minutes"] == "No") & df["minutes_spent_clean"].notna()].copy()

overall = valid.groupby("execution_mode", as_index=False).agg(
    n=("minutes_spent_clean","count"),
    mean_minutes=("minutes_spent_clean","mean"),
    median_minutes=("minutes_spent_clean","median"),
)
print("OVERALL")
print(overall.to_string(index=False))

plt.figure(figsize=(7,4))
plt.bar(overall["execution_mode"], overall["median_minutes"])
plt.xlabel("Execution mode")
plt.ylabel("Median minutes spent")
plt.title("Median task time by execution mode")
plt.tight_layout(); plt.show()

by_task = valid.groupby(["task_type","execution_mode"], as_index=False).agg(
    n=("minutes_spent_clean","count"), median_minutes=("minutes_spent_clean","median")
)
print("\nBY TASK TYPE AND MODE")
print(by_task.to_string(index=False))

# Plot one task type separately to avoid mixing units/composition.
focus="Data Entry"
sub=by_task[by_task["task_type"]==focus]
plt.figure(figsize=(7,4))
plt.bar(sub["execution_mode"], sub["median_minutes"])
plt.xlabel("Execution mode")
plt.ylabel("Median minutes spent")
plt.title(f"{focus}: median time by execution mode")
plt.tight_layout(); plt.show()

## Problem 3.45.8 — Human review, errors and outcomes

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/automation_pilot_cleaned.csv"
df = pd.read_csv(url)
df["errors_found_clean"] = pd.to_numeric(df["errors_found_clean"], errors="coerce")

# Focus on workflows in which review is especially relevant.
sub = df[df["execution_mode"].isin(["Automated","AI-assisted"])].copy()
sub["review_indicator"] = sub["human_review_clean"].map({"Yes":1,"No":0})

review_rate = sub.groupby("execution_mode", as_index=False).agg(
    n=("review_indicator","count"), review_rate=("review_indicator","mean")
)
review_rate["review_rate_pct"] = review_rate["review_rate"]*100
print("REVIEW RATE")
print(review_rate)

error_summary = sub.groupby("human_review_clean", as_index=False).agg(
    n=("errors_found_clean","count"),
    mean_errors=("errors_found_clean","mean"),
    median_errors=("errors_found_clean","median"),
)
print("\nERRORS BY REVIEW STATUS")
print(error_summary)

outcomes = pd.crosstab(sub["human_review_clean"], sub["outcome"], normalize="index")*100
print("\nOUTCOME PERCENTAGES WITHIN REVIEW STATUS")
print(outcomes.round(1))

plt.figure(figsize=(7,4))
plt.bar(review_rate["execution_mode"], review_rate["review_rate_pct"])
plt.xlabel("Execution mode")
plt.ylabel("Human review rate (%)")
plt.title("Human review in automated and AI-assisted tasks")
plt.ylim(0,100)
plt.tight_layout(); plt.show()

print("\nHIGH-ERROR / NO-REVIEW FLAGGED CASES")
print(sub.loc[sub["flag_high_error_no_review"] == "Yes", ["task_id","task_type","execution_mode","errors_found_clean","outcome","comment_raw"]].to_string(index=False))

## Problem 3.45.9 — Recurring language in staff comments

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied/automation_pilot_cleaned.csv"
df = pd.read_csv(url)
df["comment_clean"] = df["comment_clean"].fillna("").astype(str)

df["has_comment"] = df["comment_clean"].str.strip().ne("")
availability = df.groupby("execution_mode").agg(records=("task_id","count"), comments=("has_comment","sum"))
availability["availability_pct"] = availability["comments"]/availability["records"]*100
print("COMMENT AVAILABILITY")
print(availability)

all_tokens=" ".join(df.loc[df["has_comment"],"comment_clean"]).split()
top=pd.DataFrame(Counter(all_tokens).most_common(15),columns=["word","count"])
print("\nTOP WORDS")
print(top)

plt.figure(figsize=(8,5))
pt=top.sort_values("count")
plt.barh(pt["word"],pt["count"])
plt.xlabel("Word count")
plt.ylabel("Prepared word")
plt.title("Frequent words in staff automation-pilot comments")
plt.tight_layout(); plt.show()

# Normalised top terms by execution mode.
for mode in ["Manual","Automated","AI-assisted"]:
    text=" ".join(df.loc[(df["execution_mode"]==mode)&df["has_comment"],"comment_clean"])
    tokens=text.split(); counts=Counter(tokens); total=len(tokens)
    table=pd.DataFrame(counts.most_common(10),columns=["word","count"])
    table["per_1000_tokens"] = table["count"]/total*1000 if total else 0
    print(f"\n{mode}")
    print(table)

terms=["review","trust","error","failed","human","faster","not"]
pattern="|".join(terms)
examples=df[df["comment_raw"].fillna("").str.lower().str.contains(pattern,regex=True)]
print("\nORIGINAL COMMENT EXAMPLES")
print(examples[["execution_mode","staff_sentiment","comment_raw"]].head(30).to_string(index=False))

## Problem 3.45.10 — Cross-dataset city-month evidence table

The model solution uses all three curated datasets. Each dataset is aggregated independently to city-month before joining.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

base = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/applied"
service = pd.read_csv(base + "/service_requests_cleaned.csv")
support = pd.read_csv(base + "/digital_support_sessions_cleaned.csv")
auto = pd.read_csv(base + "/automation_pilot_cleaned.csv")

# Ensure relevant measures are numeric after CSV loading.
service["satisfaction_score_clean"] = pd.to_numeric(service["satisfaction_score_clean"], errors="coerce")
support["wait_minutes_clean"] = pd.to_numeric(support["wait_minutes_clean"], errors="coerce")
support = support.copy()
auto["minutes_spent_clean"] = pd.to_numeric(auto["minutes_spent_clean"], errors="coerce")
auto["errors_found_clean"] = pd.to_numeric(auto["errors_found_clean"], errors="coerce")

# Create simple indicators used in aggregate rates.
support["resolved_indicator"] = support["resolved_clean"].map({"Yes":1,"No":0})
auto["review_indicator"] = auto["human_review_clean"].map({"Yes":1,"No":0})

service_m = service.groupby(["city","report_month"], as_index=False).agg(
    request_count=("request_id","count"),
    mean_service_satisfaction=("satisfaction_score_clean","mean"),
    valid_service_satisfaction_n=("satisfaction_score_clean","count"),
)

support_m = support.groupby(["city","report_month"], as_index=False).agg(
    support_sessions=("session_id","count"),
    median_wait_minutes=("wait_minutes_clean","median"),
    support_resolution_rate=("resolved_indicator","mean"),
)

auto_m = auto.groupby(["city","report_month"], as_index=False).agg(
    pilot_tasks=("task_id","count"),
    median_task_minutes=("minutes_spent_clean","median"),
    human_review_rate=("review_indicator","mean"),
    mean_errors=("errors_found_clean","mean"),
)

# Verify the key is unique before any merge.
for name, table in [("service",service_m),("support",support_m),("automation",auto_m)]:
    duplicates = table.duplicated(["city","report_month"]).sum()
    print(name, "duplicate city-month keys:", int(duplicates))

combined = service_m.merge(support_m, on=["city","report_month"], how="outer")
combined = combined.merge(auto_m, on=["city","report_month"], how="outer")
combined = combined.sort_values(["city","report_month"])
print("\nCOMBINED EVIDENCE TABLE")
print(combined.head(25).to_string(index=False))

# Plot separate indicators because the units are different.
city="Copenhagen"
city_df=combined[combined["city"]==city]

plt.figure(figsize=(8,4))
plt.plot(city_df["report_month"], city_df["request_count"], marker="o")
plt.xlabel("Reporting month"); plt.ylabel("Service requests")
plt.title(f"{city}: municipal service request volume")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(city_df["report_month"], city_df["support_sessions"], marker="o")
plt.xlabel("Reporting month"); plt.ylabel("Digital-support sessions")
plt.title(f"{city}: community digital-support activity")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

plt.figure(figsize=(8,4))
plt.plot(city_df["report_month"], city_df["human_review_rate"]*100, marker="o")
plt.xlabel("Reporting month"); plt.ylabel("Human-review rate (%)")
plt.title(f"{city}: human review in automation-pilot tasks")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# One descriptive association. This is not a causal estimate.
assoc = combined[["support_sessions","mean_service_satisfaction"]].dropna().corr().iloc[0,1]
print("\nDescriptive correlation between support-session volume and service satisfaction:", round(assoc,3))
print("Interpretation warning: shared time/city patterns and unmeasured factors can produce associations.")